# Notebook 06 — Repurchase Prediction

## Brazilian E-Commerce Public Dataset by Olist

### Objetivo

Construir um modelo **out-of-time** para estimar a probabilidade de um cliente realizar uma segunda compra em até **90 dias** após sua primeira compra entregue.

O notebook prioriza rigor analítico e utilidade de negócio:

- evita **data leakage**;
- utiliza apenas sinais originados na primeira compra entregue;
- usa **split temporal por coorte mensal**;
- trata o forte desbalanceamento com métricas adequadas a evento raro;
- compara Logistic Regression e Random Forest contra baseline Dummy;
- converte scores em ranking operacional para CRM;
- avalia estabilidade temporal, calibração e importância das features.

> **Target:** `repurchase_90d = 1` quando a segunda compra entregue acontece até 90 dias após a primeira compra entregue.

> **Momento operacional do score:** após a conclusão da primeira compra entregue.


## 1. Configuração e bibliotecas

In [ ]:
from pathlib import Path
import json
import joblib
import sklearn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    brier_score_loss,
    precision_recall_curve,
    roc_curve,
)
from sklearn.calibration import calibration_curve
from sklearn.base import clone
from sklearn.inspection import permutation_importance

pd.set_option("display.max_columns", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

RANDOM_STATE = 42

PROJECT_ROOT = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODEL_DIR = PROJECT_ROOT / "models"

MODEL_DIR.mkdir(parents=True, exist_ok=True)

model_path = PROCESSED_DIR / "repurchase_90d_model_base.csv"

if not model_path.exists():
    raise FileNotFoundError(
        "repurchase_90d_model_base.csv não encontrado. "
        "Execute primeiro o Notebook 04 revisado."
    )

print("PROJECT_ROOT:", PROJECT_ROOT.resolve())
print("PROCESSED_DIR:", PROCESSED_DIR.resolve())
print("MODEL_DIR:", MODEL_DIR.resolve())


## 2. Carregamento da base de modelagem

In [ ]:
model_base = pd.read_csv(
    model_path,
    parse_dates=["first_purchase"]
)

required_columns = {
    "customer_unique_id",
    "first_order_id",
    "first_purchase",
    "repurchase_90d",
    "primary_payment_type",
    "installment_max",
    "dominant_category",
}

missing = required_columns.difference(model_base.columns)

if missing:
    raise ValueError(
        f"Colunas obrigatórias ausentes na model base: {sorted(missing)}"
    )

model_base["repurchase_90d"] = (
    model_base["repurchase_90d"]
    .astype(int)
)

model_base = (
    model_base
    .sort_values("first_purchase")
    .reset_index(drop=True)
)

model_base["first_purchase_month_of_year"] = (
    model_base["first_purchase"].dt.month
)

model_base["acquisition_month"] = (
    model_base["first_purchase"]
    .dt.to_period("M")
)

model_base["credit_card_installments"] = np.where(
    model_base["primary_payment_type"].eq("credit_card"),
    model_base["installment_max"],
    np.nan,
)

print("Model base:", model_base.shape)
print(
    "Período:",
    model_base["first_purchase"].min(),
    "→",
    model_base["first_purchase"].max(),
)
print(
    "Target rate:",
    f"{model_base['repurchase_90d'].mean():.2%}"
)

display(model_base.head())


## 3. Sanity checks e perfil do target

In [ ]:
assert model_base["customer_unique_id"].is_unique
assert model_base["first_order_id"].is_unique
assert model_base["repurchase_90d"].isin([0, 1]).all()
assert model_base["first_purchase"].notna().all()
assert model_base["first_purchase"].is_monotonic_increasing

class_summary = (
    model_base["repurchase_90d"]
    .value_counts()
    .rename_axis("repurchase_90d")
    .reset_index(name="customers")
)

class_summary["share"] = (
    class_summary["customers"]
    / class_summary["customers"].sum()
)

display(class_summary)

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(
    class_summary["repurchase_90d"].astype(str),
    class_summary["share"] * 100,
)
ax.set_title("Distribuição do target — recompra em 90 dias")
ax.set_xlabel("repurchase_90d")
ax.set_ylabel("Clientes (%)")

plt.tight_layout()
plt.show()


### Interpretação

A recompra em 90 dias é um **evento raro**.

Nesse cenário:

- acurácia isolada é pouco informativa;
- PR-AUC é mais útil para avaliar a classe positiva;
- Lift e Recall@K são especialmente relevantes para campanhas.


## 4. Definição das features

In [ ]:
numeric_features = [
    "first_order_payment_value",
    "first_order_product_value",
    "first_order_freight_value",
    "first_order_item_count",
    "first_order_distinct_products",
    "first_order_distinct_sellers",
    "freight_ratio",
    "payment_record_count",
    "payment_type_count",
    "first_purchase_month_of_year",
    "first_purchase_hour",
    "credit_card_installments",
]

categorical_features = [
    "primary_payment_type",
    "dominant_category",
    "first_purchase_dow",
]

feature_columns = (
    numeric_features
    + categorical_features
)

missing_features = [
    c for c in feature_columns
    if c not in model_base.columns
]

if missing_features:
    raise ValueError(
        f"Features ausentes na model base: {missing_features}"
    )

display(
    model_base[feature_columns]
    .isna()
    .sum()
    .to_frame("nulls")
    .sort_values(
        "nulls",
        ascending=False,
    )
)


### Observação metodológica

O mês absoluto de aquisição (`YYYY-MM`) **não entra como feature**, evitando que o modelo simplesmente memorize tendência temporal.

Mantemos apenas:

- mês do ano;
- dia da semana;
- hora da compra;

como sinais de sazonalidade.


## 5. Split temporal por coorte mensal

In [ ]:
months = (
    model_base["acquisition_month"]
    .drop_duplicates()
    .sort_values()
    .tolist()
)

if len(months) < 6:
    raise ValueError(
        "Poucas coortes mensais para split temporal robusto."
    )

train_cut = max(
    1,
    int(len(months) * 0.70)
)

valid_cut = max(
    train_cut + 1,
    int(len(months) * 0.85)
)

valid_cut = min(
    valid_cut,
    len(months) - 1,
)

train_months = months[:train_cut]
valid_months = months[train_cut:valid_cut]
test_months = months[valid_cut:]

train_df = model_base.loc[
    model_base["acquisition_month"].isin(train_months)
].copy()

valid_df = model_base.loc[
    model_base["acquisition_month"].isin(valid_months)
].copy()

test_df = model_base.loc[
    model_base["acquisition_month"].isin(test_months)
].copy()

split_summary = pd.DataFrame([
    {
        "split": "train",
        "customers": len(train_df),
        "start": train_df["first_purchase"].min(),
        "end": train_df["first_purchase"].max(),
        "start_month": str(min(train_months)),
        "end_month": str(max(train_months)),
        "repurchase_rate_90d": train_df["repurchase_90d"].mean(),
    },
    {
        "split": "validation",
        "customers": len(valid_df),
        "start": valid_df["first_purchase"].min(),
        "end": valid_df["first_purchase"].max(),
        "start_month": str(min(valid_months)),
        "end_month": str(max(valid_months)),
        "repurchase_rate_90d": valid_df["repurchase_90d"].mean(),
    },
    {
        "split": "test",
        "customers": len(test_df),
        "start": test_df["first_purchase"].min(),
        "end": test_df["first_purchase"].max(),
        "start_month": str(min(test_months)),
        "end_month": str(max(test_months)),
        "repurchase_rate_90d": test_df["repurchase_90d"].mean(),
    },
])

display(split_summary)


In [ ]:
assert (
    train_df["first_purchase"].max()
    < valid_df["first_purchase"].min()
)

assert (
    valid_df["first_purchase"].max()
    < test_df["first_purchase"].min()
)

assert set(train_months).isdisjoint(valid_months)
assert set(train_months).isdisjoint(test_months)
assert set(valid_months).isdisjoint(test_months)


## 6. Pré-processamento

In [ ]:
def make_preprocessor():
    numeric_pipe = Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            ),
        ),
        (
            "scaler",
            StandardScaler(),
        ),
    ])

    categorical_pipe = Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            ),
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                min_frequency=50,
            ),
        ),
    ])

    return ColumnTransformer([
        (
            "num",
            numeric_pipe,
            numeric_features,
        ),
        (
            "cat",
            categorical_pipe,
            categorical_features,
        ),
    ])


X_train = train_df[feature_columns]
y_train = train_df["repurchase_90d"]

X_valid = valid_df[feature_columns]
y_valid = valid_df["repurchase_90d"]

X_test = test_df[feature_columns]
y_test = test_df["repurchase_90d"]


## 7. Métricas para evento raro

In [ ]:
def top_k_metrics(
    y_true,
    y_score,
    k,
):
    y_true = np.asarray(y_true)
    y_score = np.asarray(y_score)

    n_top = max(
        1,
        int(
            np.ceil(
                len(y_true) * k
            )
        ),
    )

    idx = (
        np.argsort(y_score)[::-1]
        [:n_top]
    )

    prevalence = y_true.mean()

    precision_k = (
        y_true[idx].mean()
    )

    recall_k = (
        y_true[idx].sum()
        / y_true.sum()
        if y_true.sum()
        else np.nan
    )

    lift_k = (
        precision_k / prevalence
        if prevalence > 0
        else np.nan
    )

    return (
        precision_k,
        recall_k,
        lift_k,
    )


def evaluate_scores(
    y_true,
    y_score,
    model_name,
    split_name,
):
    y_true = np.asarray(y_true)
    y_score = np.asarray(y_score)

    prevalence = y_true.mean()

    result = {
        "model": model_name,
        "split": split_name,
        "customers": len(y_true),
        "positives": int(y_true.sum()),
        "prevalence": prevalence,
        "pr_auc": average_precision_score(
            y_true,
            y_score,
        ),
        "roc_auc": roc_auc_score(
            y_true,
            y_score,
        )
        if len(np.unique(y_true)) == 2
        else np.nan,
        "brier_score": brier_score_loss(
            y_true,
            y_score,
        ),
    }

    for k in [
        0.05,
        0.10,
        0.20,
    ]:
        precision_k, recall_k, lift_k = (
            top_k_metrics(
                y_true,
                y_score,
                k,
            )
        )

        suffix = int(k * 100)

        result[
            f"precision_at_{suffix}pct"
        ] = precision_k

        result[
            f"recall_at_{suffix}pct"
        ] = recall_k

        result[
            f"lift_at_{suffix}pct"
        ] = lift_k

    result[
        "pr_auc_over_prevalence"
    ] = (
        result["pr_auc"] / prevalence
        if prevalence > 0
        else np.nan
    )

    return result


## 8. Modelos candidatos

In [ ]:
candidates = {
    "Dummy Prior":
        DummyClassifier(
            strategy="prior",
        ),

    "Logistic Regression":
        LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            random_state=RANDOM_STATE,
        ),

    "Random Forest":
        RandomForestClassifier(
            n_estimators=400,
            min_samples_leaf=20,
            class_weight="balanced_subsample",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
}

validation_rows = []
validation_scores = {}
fitted_candidates = {}

for name, estimator in candidates.items():
    pipe = Pipeline([
        (
            "preprocessor",
            make_preprocessor(),
        ),
        (
            "model",
            estimator,
        ),
    ])

    pipe.fit(
        X_train,
        y_train,
    )

    score = (
        pipe
        .predict_proba(
            X_valid
        )[:, 1]
    )

    validation_scores[name] = score
    fitted_candidates[name] = pipe

    validation_rows.append(
        evaluate_scores(
            y_valid,
            score,
            name,
            "validation",
        )
    )

validation_metrics = (
    pd.DataFrame(
        validation_rows
    )
    .sort_values(
        [
            "pr_auc",
            "lift_at_10pct",
            "brier_score",
        ],
        ascending=[
            False,
            False,
            True,
        ],
    )
)

display(validation_metrics)


## 9. Curvas Precision–Recall e ROC na validação

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

for name, score in validation_scores.items():
    precision, recall, _ = (
        precision_recall_curve(
            y_valid,
            score,
        )
    )

    ax.plot(
        recall,
        precision,
        label=name,
    )

ax.axhline(
    y_valid.mean(),
    linestyle="--",
    label="Prevalência",
)

ax.set_title(
    "Precision–Recall — validação temporal"
)
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.legend()

plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

for name, score in validation_scores.items():
    if y_valid.nunique() == 2:
        fpr, tpr, _ = (
            roc_curve(
                y_valid,
                score,
            )
        )

        ax.plot(
            fpr,
            tpr,
            label=name,
        )

ax.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
)

ax.set_title(
    "ROC — validação temporal"
)
ax.set_xlabel(
    "False Positive Rate"
)
ax.set_ylabel(
    "True Positive Rate"
)
ax.legend()

plt.tight_layout()
plt.show()


## 10. Seleção do modelo

In [ ]:
eligible_models = (
    validation_metrics.loc[
        validation_metrics[
            "model"
        ].ne("Dummy Prior")
    ]
    .copy()
)

eligible_models = (
    eligible_models
    .sort_values(
        [
            "pr_auc",
            "lift_at_10pct",
            "brier_score",
        ],
        ascending=[
            False,
            False,
            True,
        ],
    )
)

best_model_name = (
    eligible_models.iloc[0]["model"]
)

print(
    "Modelo selecionado:",
    best_model_name
)

print(
    "PR-AUC validação:",
    f"{eligible_models.iloc[0]['pr_auc']:.4f}"
)

print(
    "Lift@10% validação:",
    f"{eligible_models.iloc[0]['lift_at_10pct']:.2f}x"
)


## 11. Refit em treino + validação e teste final out-of-time

In [ ]:
train_valid_df = pd.concat(
    [
        train_df,
        valid_df,
    ],
    ignore_index=True,
)

X_train_valid = (
    train_valid_df[
        feature_columns
    ]
)

y_train_valid = (
    train_valid_df[
        "repurchase_90d"
    ]
)

best_estimator = clone(
    candidates[
        best_model_name
    ]
)

final_model = Pipeline([
    (
        "preprocessor",
        make_preprocessor(),
    ),
    (
        "model",
        best_estimator,
    ),
])

final_model.fit(
    X_train_valid,
    y_train_valid,
)

test_score = (
    final_model
    .predict_proba(
        X_test
    )[:, 1]
)

test_metrics = pd.DataFrame([
    evaluate_scores(
        y_test,
        test_score,
        best_model_name,
        "test",
    )
])

display(test_metrics)


## 12. Estabilidade temporal — validação × teste

In [ ]:
stability = pd.concat(
    [
        validation_metrics.loc[
            validation_metrics[
                "model"
            ].eq(
                best_model_name
            )
        ],
        test_metrics,
    ],
    ignore_index=True,
)

display(
    stability[
        [
            "model",
            "split",
            "prevalence",
            "pr_auc",
            "roc_auc",
            "brier_score",
            "lift_at_10pct",
            "recall_at_10pct",
            "precision_at_10pct",
        ]
    ]
)

fig, ax = plt.subplots(figsize=(7, 4))

ax.bar(
    stability["split"],
    stability["pr_auc"],
)

ax.set_title(
    f"PR-AUC — estabilidade temporal ({best_model_name})"
)
ax.set_ylabel(
    "PR-AUC"
)

plt.tight_layout()
plt.show()


## 13. Calibração

In [ ]:
prob_true, prob_pred = (
    calibration_curve(
        y_test,
        test_score,
        n_bins=8,
        strategy="quantile",
    )
)

fig, ax = plt.subplots(figsize=(6, 6))

ax.plot(
    prob_pred,
    prob_true,
    marker="o",
    label=best_model_name,
)

limit = max(
    prob_pred.max(),
    prob_true.max(),
)

ax.plot(
    [0, limit],
    [0, limit],
    linestyle="--",
    label="Calibração perfeita",
)

ax.set_title(
    "Curva de calibração — teste temporal"
)
ax.set_xlabel(
    "Probabilidade média prevista"
)
ax.set_ylabel(
    "Frequência observada"
)
ax.legend()

plt.tight_layout()
plt.show()

calibration_table = pd.DataFrame({
    "mean_predicted_probability":
        prob_pred,
    "observed_repurchase_rate":
        prob_true,
})

display(calibration_table)


## 14. Priorização por decis de score

In [ ]:
scored_test = (
    test_df[
        [
            "customer_unique_id",
            "first_order_id",
            "first_purchase",
            "repurchase_90d",
            "first_order_payment_value",
            "primary_payment_type",
            "dominant_category",
        ]
    ]
    .copy()
)

scored_test[
    "repurchase_score"
] = test_score

scored_test[
    "score_decile"
] = (
    pd.qcut(
        scored_test[
            "repurchase_score"
        ]
        .rank(
            method="first",
            ascending=False,
        ),
        q=10,
        labels=False,
    )
    + 1
)

baseline_rate = (
    scored_test[
        "repurchase_90d"
    ].mean()
)

decile_summary = (
    scored_test
    .groupby(
        "score_decile"
    )
    .agg(
        customers=(
            "customer_unique_id",
            "size",
        ),
        repurchasers=(
            "repurchase_90d",
            "sum",
        ),
        observed_repurchase_rate=(
            "repurchase_90d",
            "mean",
        ),
        avg_score=(
            "repurchase_score",
            "mean",
        ),
        avg_first_ticket=(
            "first_order_payment_value",
            "mean",
        ),
    )
    .reset_index()
    .sort_values(
        "score_decile"
    )
)

decile_summary[
    "lift_vs_average"
] = (
    decile_summary[
        "observed_repurchase_rate"
    ]
    / baseline_rate
)

display(decile_summary)


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

ax.bar(
    decile_summary[
        "score_decile"
    ].astype(str),
    decile_summary[
        "observed_repurchase_rate"
    ] * 100,
)

ax.axhline(
    baseline_rate * 100,
    linestyle="--",
    label="Média do teste",
)

ax.set_title(
    "Taxa observada de recompra por decil de score"
)
ax.set_xlabel(
    "Decil de score — 1 = maior propensão"
)
ax.set_ylabel(
    "Recompra observada em 90 dias (%)"
)
ax.legend()

plt.tight_layout()
plt.show()


## 15. Simulação de campanha por orçamento

In [ ]:
campaign_rows = []

for k in [
    0.05,
    0.10,
    0.20,
    0.30,
]:
    p, r, l = (
        top_k_metrics(
            y_test,
            test_score,
            k,
        )
    )

    n_target = max(
        1,
        int(
            np.ceil(
                len(test_df) * k
            )
        ),
    )

    campaign_rows.append({
        "target_share":
            k,
        "target_customers":
            n_target,
        "precision_in_target":
            p,
        "recall_of_repurchasers":
            r,
        "lift_vs_random":
            l,
        "expected_repurchasers_observed":
            int(
                round(
                    p * n_target
                )
            ),
    })

campaign_summary = pd.DataFrame(
    campaign_rows
)

display(campaign_summary)


## 16. Capture curve

In [ ]:
capture_curve = (
    campaign_summary[
        [
            "target_share",
            "recall_of_repurchasers",
            "lift_vs_random",
        ]
    ]
    .copy()
)

fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(
    capture_curve[
        "target_share"
    ] * 100,
    capture_curve[
        "recall_of_repurchasers"
    ] * 100,
    marker="o",
)

ax.plot(
    [0, 100],
    [0, 100],
    linestyle="--",
    label="Seleção aleatória",
)

ax.set_title(
    "Curva de captura de repurchasers"
)
ax.set_xlabel(
    "Base priorizada (%)"
)
ax.set_ylabel(
    "Repurchasers capturados (%)"
)
ax.legend()

plt.tight_layout()
plt.show()

capture_curve


## 17. Importância das features

A importância nativa do modelo é mantida como diagnóstico, mas a interpretação principal usa **permutation importance** calculada sobre o teste.

Vantagens:

- mede impacto na métrica de interesse (`average_precision`);
- trabalha com as features originais;
- reduz o viés da impurity importance da Random Forest.

> Importância preditiva não implica causalidade.


In [ ]:
preprocessor = (
    final_model
    .named_steps[
        "preprocessor"
    ]
)

model = (
    final_model
    .named_steps[
        "model"
    ]
)

feature_names = (
    preprocessor
    .get_feature_names_out()
)

if hasattr(
    model,
    "feature_importances_",
):
    importance_values = (
        model.feature_importances_
    )
    importance_type = (
        "feature_importance"
    )

elif hasattr(
    model,
    "coef_",
):
    importance_values = (
        np.abs(
            model.coef_[0]
        )
    )
    importance_type = (
        "abs_coefficient"
    )

else:
    importance_values = (
        np.full(
            len(feature_names),
            np.nan,
        )
    )
    importance_type = (
        "importance"
    )

feature_importance = pd.DataFrame({
    "feature":
        feature_names,
    importance_type:
        importance_values,
}).sort_values(
    importance_type,
    ascending=False,
)

display(
    feature_importance.head(25)
)


In [ ]:
perm = permutation_importance(
    final_model,
    X_test,
    y_test,
    scoring="average_precision",
    n_repeats=10,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

permutation_df = pd.DataFrame({
    "feature":
        feature_columns,
    "importance_mean":
        perm.importances_mean,
    "importance_std":
        perm.importances_std,
}).sort_values(
    "importance_mean",
    ascending=False,
)

display(
    permutation_df
)


In [ ]:
top_perm = (
    permutation_df
    .head(15)
    .sort_values(
        "importance_mean"
    )
)

fig, ax = plt.subplots(figsize=(10, 7))

ax.barh(
    top_perm["feature"],
    top_perm["importance_mean"],
)

ax.set_title(
    f"Permutation importance — {best_model_name}"
)
ax.set_xlabel(
    "Queda média em Average Precision"
)
ax.set_ylabel(
    "Feature"
)

plt.tight_layout()
plt.show()


## 18. Performance mensal no período de teste

In [ ]:
MIN_MONTHLY_POSITIVES = 10

monthly_test = (
    scored_test.copy()
)

monthly_test[
    "first_purchase_month"
] = (
    monthly_test[
        "first_purchase"
    ]
    .dt.to_period("M")
    .astype(str)
)

monthly_rows = []

for month, g in (
    monthly_test
    .groupby(
        "first_purchase_month"
    )
):
    row = {
        "first_purchase_month":
            month,
        "customers":
            len(g),
        "repurchasers":
            int(
                g[
                    "repurchase_90d"
                ].sum()
            ),
        "repurchase_rate":
            g[
                "repurchase_90d"
            ].mean(),
        "avg_score":
            g[
                "repurchase_score"
            ].mean(),
    }

    if (
        g[
            "repurchase_90d"
        ].nunique() == 2
        and row[
            "repurchasers"
        ] >= MIN_MONTHLY_POSITIVES
    ):
        row["pr_auc"] = (
            average_precision_score(
                g["repurchase_90d"],
                g["repurchase_score"],
            )
        )

        row["roc_auc"] = (
            roc_auc_score(
                g["repurchase_90d"],
                g["repurchase_score"],
            )
        )

        _, _, lift10 = (
            top_k_metrics(
                g["repurchase_90d"],
                g["repurchase_score"],
                0.10,
            )
        )

        row["lift_at_10pct"] = (
            lift10
        )

    else:
        row["pr_auc"] = np.nan
        row["roc_auc"] = np.nan
        row["lift_at_10pct"] = np.nan

    monthly_rows.append(row)

monthly_performance = pd.DataFrame(
    monthly_rows
)

display(monthly_performance)


## 19. Interpretação executiva

### Como usar o modelo

O score deve ser usado como **ranking de priorização**, não como certeza individual de recompra.

Em uma operação real, os clientes do topo do ranking poderiam receber:

- incentivo para segunda compra;
- cross-sell baseado na categoria inicial;
- comunicação na janela de 30–90 dias;
- benefício diferenciado conforme ticket e comportamento de pagamento.

### O que o modelo não prova

- não mede efeito causal de campanha;
- não diz que uma feature causa recompra;
- não substitui teste A/B ou uplift modeling;
- não deve ser avaliado apenas por acurácia.

### Governança

Quedas de PR-AUC e Lift ao longo do tempo podem indicar drift e necessidade de retreino.


## 20. KPIs executivos do modelo

In [ ]:
best_val = (
    validation_metrics.loc[
        validation_metrics[
            "model"
        ].eq(
            best_model_name
        )
    ]
    .iloc[0]
)

best_test = (
    test_metrics.iloc[0]
)

top_decile_rate = (
    decile_summary.loc[
        decile_summary[
            "score_decile"
        ].eq(1),
        "observed_repurchase_rate",
    ]
    .iloc[0]
)

executive_kpis = pd.DataFrame({
    "kpi": [
        "model_base_customers",
        "model_base_repurchase_rate",
        "selected_model",
        "validation_pr_auc",
        "test_pr_auc",
        "test_pr_auc_over_baseline",
        "test_roc_auc",
        "test_brier_score",
        "test_precision_at_10pct",
        "test_recall_at_10pct",
        "test_lift_at_10pct",
        "top_decile_repurchase_rate",
    ],
    "value": [
        len(model_base),
        model_base[
            "repurchase_90d"
        ].mean(),
        best_model_name,
        best_val[
            "pr_auc"
        ],
        best_test[
            "pr_auc"
        ],
        best_test[
            "pr_auc_over_prevalence"
        ],
        best_test[
            "roc_auc"
        ],
        best_test[
            "brier_score"
        ],
        best_test[
            "precision_at_10pct"
        ],
        best_test[
            "recall_at_10pct"
        ],
        best_test[
            "lift_at_10pct"
        ],
        top_decile_rate,
    ],
})

display(executive_kpis)


## 21. Export dos resultados e do pipeline

In [ ]:
all_metrics = pd.concat(
    [
        validation_metrics,
        test_metrics,
    ],
    ignore_index=True,
)

exports = {
    "repurchase_model_metrics.csv":
        all_metrics,
    "repurchase_test_scored_customers.csv":
        scored_test,
    "repurchase_score_deciles.csv":
        decile_summary,
    "repurchase_campaign_simulation.csv":
        campaign_summary,
    "repurchase_capture_curve.csv":
        capture_curve,
    "repurchase_feature_importance.csv":
        feature_importance,
    "repurchase_permutation_importance.csv":
        permutation_df,
    "repurchase_monthly_test_performance.csv":
        monthly_performance,
    "repurchase_model_executive_kpis.csv":
        executive_kpis,
    "repurchase_model_split_summary.csv":
        split_summary,
    "repurchase_model_calibration.csv":
        calibration_table,
}

for filename, df in exports.items():
    path = (
        PROCESSED_DIR
        / filename
    )

    df.to_csv(
        path,
        index=False,
    )

    print(
        "Exportado:",
        path,
        df.shape,
    )

model_file = (
    MODEL_DIR
    / "repurchase_90d_pipeline.joblib"
)

joblib.dump(
    final_model,
    model_file,
)

print(
    "Modelo salvo:",
    model_file,
)


In [ ]:
metadata = {
    "target":
        "repurchase_90d",
    "target_definition":
        "second delivered purchase within 90 days after first delivered purchase",
    "horizon_days":
        90,
    "scoring_moment":
        "after_first_delivered_order",
    "selected_model":
        best_model_name,
    "feature_columns":
        feature_columns,
    "train_start":
        str(
            train_df[
                "first_purchase"
            ].min()
        ),
    "train_end":
        str(
            train_df[
                "first_purchase"
            ].max()
        ),
    "validation_start":
        str(
            valid_df[
                "first_purchase"
            ].min()
        ),
    "validation_end":
        str(
            valid_df[
                "first_purchase"
            ].max()
        ),
    "test_start":
        str(
            test_df[
                "first_purchase"
            ].min()
        ),
    "test_end":
        str(
            test_df[
                "first_purchase"
            ].max()
        ),
    "random_state":
        RANDOM_STATE,
    "pandas_version":
        pd.__version__,
    "numpy_version":
        np.__version__,
    "sklearn_version":
        sklearn.__version__,
}

metadata_file = (
    MODEL_DIR
    / "repurchase_90d_metadata.json"
)

metadata_file.write_text(
    json.dumps(
        metadata,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

print(
    "Metadata salva:",
    metadata_file,
)


## 22. Sanity checks finais

In [ ]:
assert (
    len(train_df)
    + len(valid_df)
    + len(test_df)
    == len(model_base)
)

assert (
    train_df[
        "first_purchase"
    ].max()
    < valid_df[
        "first_purchase"
    ].min()
)

assert (
    valid_df[
        "first_purchase"
    ].max()
    < test_df[
        "first_purchase"
    ].min()
)

assert set(
    train_df[
        "acquisition_month"
    ]
).isdisjoint(
    set(
        valid_df[
            "acquisition_month"
        ]
    )
)

assert set(
    valid_df[
        "acquisition_month"
    ]
).isdisjoint(
    set(
        test_df[
            "acquisition_month"
        ]
    )
)

assert train_df[
    "customer_unique_id"
].is_unique

assert valid_df[
    "customer_unique_id"
].is_unique

assert test_df[
    "customer_unique_id"
].is_unique

assert not (
    set(
        train_df[
            "customer_unique_id"
        ]
    )
    & set(
        test_df[
            "customer_unique_id"
        ]
    )
)

assert scored_test[
    "repurchase_score"
].between(0, 1).all()

assert (
    decile_summary[
        "score_decile"
    ].min()
    == 1
)

assert (
    decile_summary[
        "score_decile"
    ].max()
    == 10
)

assert final_model is not None

assert (
    PROCESSED_DIR
    / "repurchase_model_metrics.csv"
).exists()

assert (
    MODEL_DIR
    / "repurchase_90d_pipeline.joblib"
).exists()

assert (
    MODEL_DIR
    / "repurchase_90d_metadata.json"
).exists()

print(
    "Sanity checks concluídos com sucesso."
)


## 23. Conclusão para o relatório executivo

A previsão de recompra deve ser apresentada como um problema de **priorização**, e não de classificação binária tradicional.

Com uma taxa de recompra em 90 dias baixa, o ganho de negócio vem de **concentrar mais repurchasers entre os clientes priorizados**.

A comparação out-of-time também reforça uma mensagem de governança:

> o desempenho preditivo precisa ser monitorado ao longo do tempo.

### Próximos passos recomendados

1. incorporar sinais adicionais da experiência da primeira compra, respeitando o momento de scoring;
2. avaliar Gradient Boosting/XGBoost/LightGBM em benchmark controlado;
3. calibrar probabilidades se o score for usado literalmente como probabilidade;
4. implementar teste A/B para medir incremento real das campanhas;
5. evoluir para uplift modeling.
